# UbuntuCare Online Smoke Test

Use this notebook in Google Colab to launch a temporary UbuntuCare frontend **and** smoke-test its APIs. The demo starts only the frontend, API gateway, EMR, PostgreSQL, Redis, Keycloak, and migrations. AI services (ASR, TTS, RAG, and agent workers) are not started.

> **Temporary demo only:** Colab exposes the frontend through a public Cloudflare Quick Tunnel URL. Anyone with that URL can reach the login page. The notebook generates a one-time demo admin password. Do not use real patient data, do not share the URL/password, and stop the stack when finished. AI features will be unavailable.

Open in Colab: upload this `.ipynb` file at [colab.research.google.com](https://colab.research.google.com/) or open it from the repository with **Open in Colab**. Provide a public GitHub repository URL or upload a ZIP of the project source. Colab runs the containers; the tunnel exposes only the web frontend.

In [ ]:
%pip -q install requests

## 0. Launch the frontend and core services

Run these cells in order. Use a public GitHub repository containing the UbuntuCare source or upload a ZIP of the project. Docker-in-Colab availability depends on the selected Colab runtime. Login is through the **email/password** form; the SSO button needs a separately configured public Keycloak URL and is not enabled by this single-URL demo.

In [ ]:
from pathlib import Path
import shutil, subprocess, zipfile
if shutil.which('git') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'git'], check=True)

REPO_DIR = Path('/content/ubuntucare')
SOURCE_MODE = input('Source mode — github or upload [github]: ').strip().lower() or 'github'
if SOURCE_MODE == 'github':
    REPO_URL = input('Public GitHub repository URL: ').strip()
    BRANCH = input('Branch [main]: ').strip() or 'main'
    if not REPO_URL.startswith('https://'):
        raise ValueError('Use an HTTPS GitHub repository URL, or choose upload mode.')
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
elif SOURCE_MODE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if not zip_names:
        raise ValueError('Upload a ZIP archive of the UbuntuCare project source.')
    unpack_dir = Path('/content/ubuntucare_upload')
    unpack_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_names[0]) as archive:
        root = unpack_dir.resolve()
        for member in archive.infolist():
            destination = (unpack_dir / member.filename).resolve()
            if root not in destination.parents and destination != root:
                raise ValueError('Unsafe path in uploaded ZIP.')
        archive.extractall(unpack_dir)
    compose_files = list(unpack_dir.rglob('docker-compose.yml'))
    if not compose_files:
        raise FileNotFoundError('Could not find docker-compose.yml in the uploaded ZIP.')
    REPO_DIR = compose_files[0].parent
else:
    raise ValueError('Choose github or upload.')

if not (REPO_DIR / 'src/frontend/web/package.json').exists():
    raise FileNotFoundError(f'Not an UbuntuCare source tree: {REPO_DIR}')
print('Using source checkout:', REPO_DIR)

In [ ]:
import os, shutil, time

def run_cmd(args, **kwargs):
    return subprocess.run(args, check=True, text=True, **kwargs)

subprocess.run(['apt-get', 'update', '-qq'], check=True)
if shutil.which('docker') is None:
    run_cmd(['apt-get', 'install', '-y', '-qq', 'docker.io', 'curl', 'git'])
else:
    run_cmd(['apt-get', 'install', '-y', '-qq', 'curl', 'git'])

COMPOSE_CMD = ['docker', 'compose']
compose_check = subprocess.run(COMPOSE_CMD + ['version'], capture_output=True, text=True)
if compose_check.returncode != 0:
    plugin_dir = Path('/usr/local/lib/docker/cli-plugins')
    plugin_dir.mkdir(parents=True, exist_ok=True)
    plugin = plugin_dir / 'docker-compose'
    run_cmd(['curl', '-fsSL', 'https://github.com/docker/compose/releases/download/v2.32.4/docker-compose-linux-x86_64', '-o', str(plugin)])
    plugin.chmod(0o755)

if shutil.which('cloudflared') is None:
    run_cmd(['curl', '-fsSL', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-o', '/usr/local/bin/cloudflared'])
    Path('/usr/local/bin/cloudflared').chmod(0o755)

docker_check = subprocess.run(['docker', 'info'], capture_output=True, text=True)
if docker_check.returncode != 0:
    DOCKER_LOG_PATH = Path('/content/dockerd.log')
    DOCKER_LOG = DOCKER_LOG_PATH.open('w')
    DOCKER_DAEMON = subprocess.Popen(['dockerd', '--host=unix:///var/run/docker.sock'], stdout=DOCKER_LOG, stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(90):
        if subprocess.run(['docker', 'info'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0:
            break
        if DOCKER_DAEMON.poll() is not None:
            raise RuntimeError('Docker daemon stopped. See /content/dockerd.log; this Colab runtime may not permit nested Docker.')
        time.sleep(2)
    else:
        raise RuntimeError('Docker daemon did not become ready. See /content/dockerd.log.')

print(subprocess.run(COMPOSE_CMD + ['version'], capture_output=True, text=True).stdout.strip())
print('Docker engine is ready.')

In [ ]:
import secrets
import string
from uuid import uuid4

def make_demo_password():
    # Meet the Keycloak demo realm's minimum length/complexity policy.
    return 'Colab-7' + secrets.token_hex(12) + '!'

DEMO_LOGIN_EMAIL = 'admin@ubuntucare.gov.zm'
DEMO_LOGIN_PASSWORD = make_demo_password()
env = {
    'POSTGRES_USER': 'ubuntucare',
    'POSTGRES_DB': 'ubuntucare',
    'POSTGRES_PASSWORD': secrets.token_urlsafe(32),
    'SECRET_KEY': secrets.token_urlsafe(48),
    'KEYCLOAK_ADMIN_USER': 'admin',
    'KEYCLOAK_ADMIN_PASSWORD': make_demo_password(),
    'KEYCLOAK_DEMO_USER_PASSWORD': DEMO_LOGIN_PASSWORD,
    'KEYCLOAK_REALM': 'ubuntucare',
    'KEYCLOAK_CLIENT_ID': 'api-gateway',
    'KEYCLOAK_CLIENT_SECRET': secrets.token_urlsafe(36),
    'KEYCLOAK_FRONTEND_CLIENT_ID': 'frontend',
    'KEYCLOAK_HOSTNAME': 'keycloak',
    'MINIO_ROOT_USER': 'ubuntucare',
    'MINIO_ROOT_PASSWORD': secrets.token_urlsafe(32),
    'FLOWER_PASSWORD': secrets.token_urlsafe(24),
    'ELASTIC_PASSWORD': secrets.token_urlsafe(24),
    'GRAFANA_ADMIN_PASSWORD': secrets.token_urlsafe(24),
    'FRONTEND_PORT': '3000',
    'API_GATEWAY_PORT': '8000',
    'EMR_PORT': '8001',
}
COLAB_ENV_FILE = Path('/content/ubuntucare-colab.env')
COLAB_ENV_FILE.write_text('\n'.join(f'{key}={value}' for key, value in env.items()) + '\n', encoding='utf-8')
COLAB_ENV_FILE.chmod(0o600)
COLAB_PROJECT = f'ubuntucare-colab-{uuid4().hex[:8]}'
COMPOSE_FILES = ['-f', 'docker-compose.yml', '-f', 'docs/colab/docker-compose.colab.yml']
COMPOSE = COMPOSE_CMD + ['-p', COLAB_PROJECT, '--env-file', str(COLAB_ENV_FILE)] + COMPOSE_FILES

# Starting only these targets starts their core dependencies; Compose AI profiles remain off.
print('Building and starting frontend + API gateway + EMR and required core dependencies...')
run_cmd(COMPOSE + ['up', '-d', '--build', 'frontend', 'api-gateway', 'emr'], cwd=REPO_DIR)
print('Core stack is up. No ASR, TTS, RAG, or agent-worker services were started.')

In [ ]:
import re
from IPython.display import HTML, display

# Wait for the login page before opening a temporary HTTPS tunnel.
for _ in range(90):
    try:
        local_page = requests.get('http://127.0.0.1:3000/login', timeout=5)
        if local_page.status_code == 200:
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise RuntimeError('Frontend did not become ready. Check: docker compose logs frontend emr api-gateway')

TUNNEL_LOG_PATH = Path('/content/cloudflared.log')
TUNNEL_LOG = TUNNEL_LOG_PATH.open('w')
CLOUDFLARED = subprocess.Popen(['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:3000'], stdout=TUNNEL_LOG, stderr=subprocess.STDOUT, start_new_session=True)
FRONTEND_URL = None
for _ in range(90):
    log_text = TUNNEL_LOG_PATH.read_text(errors='ignore')
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)
    if match:
        FRONTEND_URL = match.group(0)
        break
    if CLOUDFLARED.poll() is not None:
        raise RuntimeError('Cloudflare tunnel stopped. Check /content/cloudflared.log.')
    time.sleep(2)
if not FRONTEND_URL:
    raise RuntimeError('Could not obtain a Cloudflare URL. Check /content/cloudflared.log.')
BASE_URL = FRONTEND_URL
print('Frontend login:', FRONTEND_URL + '/login')
print('Demo login email:', DEMO_LOGIN_EMAIL)
print('One-time demo password:', DEMO_LOGIN_PASSWORD)
print('Use the email/password form. Do not click SSO in this single-tunnel demo.')
display(HTML(f'<p><a href="{FRONTEND_URL}/login" target="_blank"><strong>Open UbuntuCare login</strong></a></p>'))
print('Do not save/share notebook outputs: the temporary URL and demo password are visible here.')

In [ ]:
from getpass import getpass
from datetime import datetime, timezone
from uuid import uuid4
from urllib.parse import urlparse
import requests

# If you launched the Colab stack above, reuse its temporary frontend URL.
# Otherwise point to an already-running public UbuntuCare deployment.
if 'FRONTEND_URL' not in globals():
    BASE_URL = input('Public UbuntuCare URL: ').strip().rstrip('/')
if not BASE_URL.startswith(('https://', 'http://')):
    raise ValueError('Enter a full URL starting with https:// or http://')
if urlparse(BASE_URL).hostname in {'localhost', '127.0.0.1', '::1'}:
    raise ValueError('Colab cannot reach localhost on your computer. Use a public deployment URL.')

HOSPITAL_ID = '00000000-0000-4000-8000-000000000001'
RUN_WRITE_TESTS = False  # Change to True only on a test/staging deployment.

session = requests.Session()
session.headers.update({'Accept': 'application/json'})
API = f'{BASE_URL}/api/v1'

if 'DEMO_LOGIN_PASSWORD' in globals():
    LOGIN_EMAIL, LOGIN_PASSWORD = DEMO_LOGIN_EMAIL, DEMO_LOGIN_PASSWORD
else:
    LOGIN_EMAIL = input('UbuntuCare login email: ').strip()
    LOGIN_PASSWORD = getpass('UbuntuCare password (hidden): ')
login_response = session.post(f'{API}/auth/login', json={'email': LOGIN_EMAIL, 'password': LOGIN_PASSWORD}, timeout=30)
print('POST /auth/login:', login_response.status_code)
login_response.raise_for_status()
ACCESS_TOKEN = login_response.json()['access_token']
session.headers['Authorization'] = f'Bearer {ACCESS_TOKEN}'
session.headers['X-Hospital-ID'] = HOSPITAL_ID
print('Authenticated as:', LOGIN_EMAIL)
print('Configured deployment:', BASE_URL)
print('Write tests enabled:', RUN_WRITE_TESTS)

## 1. Gateway health (read-only)

This request does not need a token. A `healthy` result means the gateway can reach its database and Redis checks; other dependencies may still need separate attention.

In [ ]:
health_url = 'http://127.0.0.1:8000/health' if 'FRONTEND_URL' in globals() else f'{BASE_URL}/health'
health_response = session.get(health_url, timeout=30)
print('GET /health:', health_response.status_code)
health_response.raise_for_status()
health = health_response.json()
print('Gateway status:', health.get('status'))
print('Database:', health.get('checks', {}).get('database', {}).get('status'))
print('Redis:', health.get('checks', {}).get('redis', {}).get('status'))

## 2. Authenticated read-only checks

The notebook logs in using the credentials above. The temporary stack uses a generated demo admin account. The notebook reports status/counts only; it avoids printing patient records.

In [ ]:
def get_json(path, params=None):
    response = session.get(f'{API}{path}', params=params, timeout=30)
    print(f'GET {path}:', response.status_code)
    if not response.ok:
        print('Response:', response.text[:500])
        response.raise_for_status()
    return response.json()

departments_data = get_json('/hospital/departments')
departments = departments_data if isinstance(departments_data, list) else departments_data.get('items', [])
active_department_names = {d.get('name') for d in departments if d.get('is_active', True)}
print('Active departments:', sorted(active_department_names))
expected_departments = {'General', 'Eye', 'Dental'}
assert active_department_names == expected_departments, f'Expected exactly {sorted(expected_departments)}, got {sorted(active_department_names)}'

staff_data = get_json('/hospital/staff', params={'per_page': 100})
staff_items = staff_data.get('items', []) if isinstance(staff_data, dict) else staff_data
print('Directory staff records available:', len(staff_items))

destinations = get_json('/patient-flow/destinations')
service_points = {item.get('code') for item in destinations.get('service_points', [])}
print('Core service destinations present:', sorted({'general', 'eye', 'dental'} & service_points))
assert {'general', 'eye', 'dental'}.issubset(service_points), 'General/Eye/Dental patient-flow destinations are missing'

# A unique synthetic query checks patient search without listing or displaying real patients.
search_marker = f'COLAB-NONEXISTENT-{uuid4().hex[:10]}'
patient_search = get_json('/emr/patients', params={'query': search_marker, 'per_page': 5})
print('Synthetic patient search completed; matches:', patient_search.get('total', 0))

notes_data = get_json('/emr/notes', params={'page': 1, 'per_page': 1})
print('Clinical-note API accessible; total notes:', notes_data.get('total', 'not reported'))
print('Read-only checks passed.')

## 3. Optional write test — disabled by default

Only continue in the temporary Colab database or a **test/staging environment**. Set `RUN_WRITE_TESTS = True` in the configuration cell, then run this cell. It creates one clearly labelled synthetic patient, an Eye visit with NHIMA/review/paid test values, and a draft eye SOAP note. These records remain in the database; the notebook does not delete records. Do not use real names, NRCs, phone numbers, or clinical information.

In [ ]:
if not RUN_WRITE_TESTS:
    print('Skipped: set RUN_WRITE_TESTS = True only after confirming this is a test/staging deployment.')
else:
    marker = uuid4().hex[:8].upper()
    patient_payload = {
        'first_name': 'ColabTest',
        'last_name': f'Synthetic{marker}',
        'date_of_birth': '1990-01-01',
        'gender': 'other',
        'phone': f'+26097{uuid4().int % 10_000_000:07d}',
        'notes': 'SYNTHETIC COLAB SMOKE TEST DATA — not a real patient.',
    }
    patient_response = session.post(f'{API}/emr/patients', json=patient_payload, timeout=30)
    print('POST /emr/patients:', patient_response.status_code)
    patient_response.raise_for_status()
    patient_id = patient_response.json()['id']

    eye_department = next((d for d in departments if d.get('name') == 'Eye' and d.get('is_active', True)), None)
    journey_payload = {
        'patient_id': patient_id,
        'priority': 'routine',
        'payment_scheme': 'nhima',
        'visit_type': 'review',
        'payment_status': 'paid',
        'scheme_number': f'COLAB-{marker}',
        'chief_complaint': 'SYNTHETIC TEST DATA ONLY — routine eye review',
        'step_type': 'consultation',
        'service_point': 'eye',
        'department_id': eye_department.get('id') if eye_department else None,
        'department_name': 'Eye',
    }
    journey_response = session.post(f'{API}/patient-flow/journeys', json=journey_payload, timeout=30)
    print('POST /patient-flow/journeys:', journey_response.status_code)
    journey_response.raise_for_status()
    journey = journey_response.json()
    print('Created synthetic journey:', journey.get('journey_number'), '|', journey.get('visit_type'), '|', journey.get('payment_status'))

    note_payload = {
        'patient_id': patient_id,
        'note_type': 'SOAP',
        'title': f'Colab Smoke Test - Eye Note {marker}',
        'subjective': 'SYNTHETIC TEST DATA ONLY — no real patient symptoms.',
        'objective': 'SYNTHETIC TEST DATA ONLY — no examination performed.',
        'assessment': 'Test record only; not a clinical diagnosis.',
        'plan': 'Test record only; no treatment.',
        'authored_by': 'Colab Smoke Test',
        'authored_datetime': datetime.now(timezone.utc).isoformat(),
        'status': 'draft',
    }
    note_response = session.post(f'{API}/emr/notes', json=note_payload, timeout=30)
    print('POST /emr/notes:', note_response.status_code)
    note_response.raise_for_status()
    print('Created synthetic draft note:', note_response.json().get('id'))
    print('Write smoke test passed. Synthetic records were left in the test database.')

## 4. Stop the temporary public demo

Run this when finished. It stops the containers and tunnel. Colab will eventually destroy its VM, but stopping explicitly closes the public URL sooner.

In [ ]:
if 'COMPOSE' in globals() and 'REPO_DIR' in globals():
    subprocess.run(COMPOSE + ['down', '--volumes', '--remove-orphans'], cwd=REPO_DIR, check=False)
if 'CLOUDFLARED' in globals() and CLOUDFLARED.poll() is None:
    CLOUDFLARED.terminate()
    try:
        CLOUDFLARED.wait(timeout=10)
    except subprocess.TimeoutExpired:
        CLOUDFLARED.kill()
print('Colab demo containers and public tunnel stopped.')

## Troubleshooting

- `401`: token missing/expired; obtain a new token and rerun the configuration cell.
- `403`: token lacks the required directory/EMR/patient-flow permissions.
- `400` mentioning hospital ID: check `HOSPITAL_ID` or use the tenant UUID assigned to your deployment.
- Connection/TLS errors: verify the deployment is publicly reachable over HTTPS and that the gateway ingress allows Colab requests.
- `RUN_WRITE_TESTS` requires a role allowed to register patients, create patient journeys, and save clinical notes.